# Session 2 — build the processed long dataset

Runs `load_long()` and `clean()` from `src/data.py`, verifies the result, and saves `data/processed/vn1_long.parquet`.

In [ ]:
# Cell 1 — setup
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.data import load_long, clean, KEY

In [ ]:
# Cell 2 — load raw long frame
SALES = "data/raw/Phase_0_Sales.csv"
PRICE = "data/raw/Phase_0_Price.csv"

df = load_long(SALES, PRICE)
print(df.shape)
df.head()

In [ ]:
# Cell 3 — structure checks
n_series = df[KEY].drop_duplicates().shape[0]
print("rows          :", len(df))
print("expected      :", 15053 * 170)
print("unique series :", n_series)
print("unique dates  :", df["date"].nunique())
print("weekday(s)    :", df["date"].dt.day_name().unique())

assert len(df) == 15053 * 170, "row count off — check the melt/merge"
assert df["date"].nunique() == 170, "expected 170 weekly dates"
assert df["date"].dt.dayofweek.nunique() == 1, "dates should all fall on one weekday"

In [ ]:
# Cell 4 — sanity-check zeros vs NaN
zero_sales_pct = (df["sales"] == 0).mean()
nan_price_pct  = df["price"].isna().mean()
sold_no_price  = ((df["sales"] > 0) & (df["price"].isna())).sum()

print(f"sales == 0                      : {zero_sales_pct:.3%}")
print(f"price is NaN                    : {nan_price_pct:.3%}")
print(f"sold but no price (should be 0) : {sold_no_price}")

assert sold_no_price == 0, "found sales>0 with NaN price — investigate before cleaning"

In [ ]:
# Cell 5 — clean and report the ffill effect
before = df["price"].isna().sum()
cdf = clean(df)
after = cdf["price"].isna().sum()

print(f"price NaN before ffill : {before:,}")
print(f"price NaN after  ffill : {after:,}")
print("remaining NaN = leading pre-launch weeks (series had no sale yet)")
cdf.head()

In [ ]:
# Cell 6 — save processed long frame to parquet
OUT = "data/processed/vn1_long.parquet"
cdf.to_parquet(OUT)

check = pd.read_parquet(OUT)
print("saved     :", OUT)
print("shape     :", check.shape)
print("date dtype:", check["date"].dtype)   # expect datetime64[ns]
assert str(check["date"].dtype) == "datetime64[ns]", "date dtype not preserved"